# Diabetes Prediction: Building and Training the ANN

This notebook picks up where `01_data_preparation.ipynb` left off. All the
messy work (cleaning, feature engineering, splitting, scaling) is already
done. Here we only focus on the neural network:

1. Load the prepared data
2. Build a simple Artificial Neural Network (ANN)
3. Train it
4. Visualize how training went
5. Evaluate it on the test set
6. Save the trained model
7. Make a prediction on a new patient, with clean, simple code

Run `01_data_preparation.ipynb` first if you have not already, it creates
the CSV and JSON files this notebook loads below.


## Step 1: Import libraries

In [2]:
import json
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# import seaborn as sns

import tensorflow as tf
from tensorflow import keras

from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve)

tf.random.set_seed(42)
np.random.seed(42)


## Step 2: Load the prepared data

We train directly from the **CSV** files saved by the data preparation
notebook. Each CSV already has the scaled features and the `Outcome`
column, so all we need to do is split the target off and turn everything
into arrays.

In [4]:
train_df = pd.read_csv('diabetes_train.csv')
val_df = pd.read_csv('diabetes_val.csv')
test_df = pd.read_csv('diabetes_test.csv')

X_train = train_df.drop(columns=['Outcome']).values.astype(np.float32)
y_train = train_df['Outcome'].values.astype(np.float32)

X_val = val_df.drop(columns=['Outcome']).values.astype(np.float32)
y_val = val_df['Outcome'].values.astype(np.float32)

X_test = test_df.drop(columns=['Outcome']).values.astype(np.float32)
y_test = test_df['Outcome'].values.astype(np.float32)

print(f"Train: {X_train.shape}")
print(f"Val:   {X_val.shape}")
print(f"Test:  {X_test.shape}")
print(f"Number of features: {X_train.shape[1]}")


Train: (537, 14)
Val:   (115, 14)
Test:  (116, 14)
Number of features: 14


We also load the **JSON** metadata file. It holds the feature names, the
medians used to fill in missing values, and the scaler's numbers
(`mean` and `scale`, one per feature). We will use these later to scale a
brand new patient by hand, the same way the training data was scaled.

In [5]:
with open('diabetes_metadata.json') as f:
    metadata = json.load(f)

feature_cols = metadata['feature_cols']
train_medians = metadata['train_medians']
scaler_mean = np.array(metadata['scaler_mean'], dtype=np.float32)
scaler_scale = np.array(metadata['scaler_scale'], dtype=np.float32)

print(f"Loaded metadata for {len(feature_cols)} features")


Loaded metadata for 14 features


## Step 3: Build the ANN

Our network is a simple stack of "Dense" (fully connected) layers. Each
block does the same job in slightly more detail:

- **Dense layer**: learns a weighted combination of the inputs
- **BatchNormalization**: keeps the numbers flowing through the network in
  a stable range, this helps training go faster and more smoothly
- **ReLU activation**: lets the network learn non-linear patterns, not
  just straight lines
- **Dropout**: randomly turns off some neurons during training, this
  forces the network to not rely too heavily on any single neuron and
  helps prevent overfitting

The final layer has one neuron with a **sigmoid** activation, which
squashes the output into a probability between 0 and 1: how likely this
patient is to have diabetes.

In [6]:
def build_ann(n_features):
    inputs = keras.Input(shape=(n_features,), name='features')

    # Block 1
    x = keras.layers.Dense(128, kernel_regularizer=keras.regularizers.l2(1e-4))(inputs)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)
    x = keras.layers.Dropout(0.3)(x)

    # Block 2
    x = keras.layers.Dense(64, kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation('relu')(x)
    x = keras.layers.Dropout(0.3)(x)

    # Block 3
    x = keras.layers.Dense(32)(x)
    x = keras.layers.Activation('relu')(x)
    x = keras.layers.Dropout(0.2)(x)

    # Output: a single probability
    outputs = keras.layers.Dense(1, activation='sigmoid', name='prob')(x)

    return keras.Model(inputs, outputs, name='DiabetesANN')


model = build_ann(X_train.shape[1])
model.summary()


Model: "DiabetesANN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ features (InputLayer)                │ (None, 14)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │           1,920 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation (Activation)              │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 64)                  │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_1 (Activation)            │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_2 (Activation)            │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ prob (Dense)                         │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 13,057 (51.00 KB)

 Trainable params: 12,673 (49.50 KB)

 Non-trainable params: 384 (1.50 KB)

## Step 4: Compile the model

- **Optimizer (Adam)**: the algorithm that updates the network's weights
- **Loss (binary crossentropy)**: the standard loss function for a
  yes/no (binary) prediction problem
- **Metrics**: extra numbers we track while training, so we can see more
  than just the loss

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.AUC(name='auc'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
    ]
)


## Step 5: Train the model

A few helpers make training smarter instead of just running for a fixed
number of epochs:

- **EarlyStopping**: stops training once the validation AUC stops
  improving, and restores the best weights seen so far
- **ReduceLROnPlateau**: shrinks the learning rate if progress stalls,
  helping the model fine-tune
- **ModelCheckpoint**: saves the best version of the model to disk as we go

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_auc', patience=25,
        restore_best_weights=True, mode='max', verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_auc', factor=0.5, patience=10,
        min_lr=1e-6, mode='max', verbose=1),
    keras.callbacks.ModelCheckpoint(
        'best_diabetes.keras', monitor='val_auc',
        save_best_only=True, mode='max', verbose=0),
]

history = model.fit(
    X_train, y_train,
    epochs=300, batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1,
)

print(f"\nEpochs run:   {len(history.history['loss'])}")
print(f"Best val AUC: {max(history.history['val_auc']):.4f}")


## Step 6: Visualize training

Plotting the training history helps us see whether the model is learning
well or overfitting. If the training curve keeps improving while the
validation curve gets worse, that is a sign of overfitting.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Validation')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train')
axes[1].plot(history.history['val_accuracy'], label='Validation')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

axes[2].plot(history.history['auc'], label='Train')
axes[2].plot(history.history['val_auc'], label='Validation')
axes[2].set_title('AUC')
axes[2].set_xlabel('Epoch')
axes[2].legend()

plt.suptitle('Training History', fontsize=13)
plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()


## Step 7: Evaluate on the test set

The test set was never touched during training or tuning, so this is our
honest, final measure of how well the model performs on new patients.

In [ ]:
model.load_weights('best_diabetes.keras')

test_results = model.evaluate(X_test, y_test, verbose=0)
for name, value in zip(model.metrics_names, test_results):
    print(f"{name:<12}: {value:.4f}")


### ROC curve

The ROC curve shows the trade-off between catching true diabetes cases
(sensitivity) and avoiding false alarms, across every possible decision
threshold. The closer the curve hugs the top-left corner, the better.

In [ ]:
y_prob = model.predict(X_test, verbose=0).flatten()
auc_score = roc_auc_score(y_test, y_prob)

fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, color='crimson', label=f'ROC curve (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"ROC-AUC: {auc_score:.4f}")


### Classification report and confusion matrix

We use a plain 0.5 probability threshold to turn probabilities into
yes/no predictions, and see how the model actually did.

In [ ]:
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=['No Diabetes', 'Diabetes']))

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


## Step 8: Save the trained model

We save the model itself, plus the scaler and feature list, so that
anyone can load them later and make predictions without retraining.

In [ ]:
model.save('diabetes_ann.keras')

with open('diabetes_inference_kit.pkl', 'wb') as f:
    pickle.dump({
        'scaler': scaler,
        'feature_cols': feature_cols,
        'train_medians': train_medians,
    }, f)

print("Saved diabetes_ann.keras")
print("Saved diabetes_inference_kit.pkl")


## Step 9: Make a prediction on a new patient

This is the simple, beginner-friendly version: no extra options, no
risk levels, just three clear steps.

1. Turn the patient's measurements into the same features the model was
   trained on
2. Scale them by hand, using the `scaler_mean` and `scaler_scale` numbers
   from the JSON metadata (the same formula `StandardScaler` uses:
   `(value - mean) / scale`)
3. Ask the model for a probability, and turn it into a plain prediction

In [ ]:
def predict_patient(patient, model, feature_cols, scaler_mean, scaler_scale):
    patient = patient.copy()

    # Add the same extra features we created during data preparation
    patient['Glucose_x_BMI'] = patient['Glucose'] * patient['BMI']
    patient['Age_x_BMI'] = patient['Age'] * patient['BMI']
    patient['impaired_glucose'] = float(patient['Glucose'] >= 100)
    patient['diabetic_glucose'] = float(patient['Glucose'] >= 126)
    patient['is_obese'] = float(patient['BMI'] >= 30)
    patient['older_patient'] = float(patient['Age'] >= 40)

    # Put the features in the exact same order the model expects
    row = np.array([patient[col] for col in feature_cols], dtype=np.float32)

    # Scale by hand: same formula the training data went through
    row_scaled = (row - scaler_mean) / scaler_scale
    row_scaled = row_scaled.reshape(1, -1)

    probability = model.predict(row_scaled, verbose=0)[0][0]
    prediction = 'Diabetes' if probability >= 0.5 else 'No Diabetes'

    print(f"Probability of diabetes: {probability:.1%}")
    print(f"Prediction: {prediction}")


Let's try it on two real patients from the dataset, one known to have
diabetes and one known not to.

In [ ]:
patient_1 = {
    'Pregnancies': 6, 'Glucose': 148, 'BloodPressure': 72, 'SkinThickness': 35,
    'Insulin': 0, 'BMI': 33.6, 'DiabetesPedigreeFunction': 0.627, 'Age': 50,
}
print("Patient 1 (actual: Diabetes)")
predict_patient(patient_1, model, feature_cols, scaler_mean, scaler_scale)

print()

patient_2 = {
    'Pregnancies': 1, 'Glucose': 85, 'BloodPressure': 66, 'SkinThickness': 29,
    'Insulin': 0, 'BMI': 26.6, 'DiabetesPedigreeFunction': 0.351, 'Age': 31,
}
print("Patient 2 (actual: No Diabetes)")
predict_patient(patient_2, model, feature_cols, scaler_mean, scaler_scale)


### Done

We built, trained, evaluated, and saved an ANN for diabetes prediction,
and made predictions with clean, simple code. Try changing the patient
values above and see how the prediction changes.